# Lab 5 - Diagnose, Fix, Prove (Track N: notebook)

One file, top to bottom. Work the cells in order.

This repo has two tracks. **This notebook is Track N.** The alternative is
Track S - the script `labs/lab5/diagnose.py`. Pick one and ignore the other
for the whole lab. (You still submit `diagnose.py` + `report.md` either way.)

| Part | What you do | ~time |
|---|---|---|
| 0 | Prerequisite check (`reports/lab4.json`) | 5 min |
| A | Classify every failure. **This is most of the lab** | 50 min |
| B | Rank clusters by expected value + **write the prediction** | 20 min |
| C | Fix the top cluster. One variable at a time | 70 min |
| D | Prove it: before/after, regression check, cost | 25 min |
| Submit | Copy into `diagnose.py` + fill `report.md` | 10 min |

**Hard prerequisite:** `reports/lab4.json` exists (Lab 4 E1 `--full --save`).

**Keep open:** `labs/lab5/CONCEPTS.md`.

**The one rule:** run cells in order, and when a cell says *your turn*, edit it
there. Numbered TODOs come with scaffold - fill them; the `assert` tells you
what is missing. When done, copy classification/fix notes into
`labs/lab5/diagnose.py` if you changed it, and fill `report.md` by hand.

---

## Read this before anything else

### You are graded on the process, not the score.

When the reference ran this lab it diagnosed the dominant cluster correctly,
chose a well-motivated fix, predicted 4-6 recoveries - and measured
**-0.050 correctness**. The fix made the system worse.

**That is a complete, full-marks Lab 5.** What is graded is whether you
diagnosed before you fixed, predicted before you measured, and reported what
happened rather than what you hoped. A **-0.05 you can explain** scores above
a **+0.12 you cannot.**

### The inversion (read twice)

Gold context **fixing** an answer means **RETRIEVAL** was at fault - the
generator was fine and never got the material. Still wrong with gold context
means **generation**. People get this backwards; a whole lab's work can hang
on it.


## 0 - Setup


In [ ]:
# Make the shared library importable and load settings. Safe to re-run.
import os, sys, json, re
from pathlib import Path
from collections import Counter

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "aip").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "aip").is_dir():
    ROOT = Path.cwd().parents[1]          # notebook opened from labs/lab5
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
os.chdir(ROOT)
print("repo root:", ROOT)


In [ ]:
# Hard prerequisite: Lab 4 output.
lab4_path = ROOT / "reports/lab4.json"
assert lab4_path.exists(), (
    "reports/lab4.json not found. Finish Lab 4 first: "
    "python labs/lab4/evaluate.py --full --save reports/lab4.json"
)
lab4_rows = json.loads(lab4_path.read_text(encoding="utf-8"))
failures = [r for r in lab4_rows
            if r.get("correctness", 2) < 2 or not r.get("citations_valid", True)]
print(f"lab4.json: {len(lab4_rows)} rows, {len(failures)} failures")
print("failure ids:", [r["id"] for r in failures])

from labs.lab3.search import load_corpus, load_questions
from labs.lab5.diagnose import MODES, classify, pareto, answer_in_corpus
corpus = load_corpus()
questions = {q["id"]: q for q in load_questions(include_unanswerable=True)}
print("modes:", MODES)


## Part A - Classify every failure (50 min)

**Why.** *"RAG is bad"* is not a diagnosis. Exactly one stage failed on each
wrong answer. This part is most of the lab, and the rest depends on it.

The script automates six of seven modes. **Mode 2 needs your eyes.**

> **Checkpoint.** A tally spread evenly across all seven modes means you
> mis-classified - re-check the mode-6 test. A heavily concentrated tally is
> **normal**. Yours will not match the reference's; the distribution is a
> property of *your* pipeline.


In [ ]:
# A1 - run the classifier. LLM probes cost a few cents; --no-llm for offline triage.
# Produced tally + evidence per failure; saves reports/lab5_diagnosis.json.
import subprocess

cmd = [sys.executable, str(ROOT / "labs/lab5/diagnose.py"),
       "--input", "reports/lab4.json",
       "--save", "reports/lab5_diagnosis.json"]
print(" ".join(cmd))
# Uncomment to run (several minutes if probes are on):
# result = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
# print(result.stdout[-4000:])
# print(result.stderr[-2000:] if result.stderr else "")

# Or load a diagnosis that already exists:
diag_path = ROOT / "reports/lab5_diagnosis.json"
if diag_path.exists():
    diag = json.loads(diag_path.read_text(encoding="utf-8"))
    tally = Counter(d["mode"] for d in diag)
    print(f"\nloaded {len(diag)} classified failures")
    print(pareto(tally))
else:
    diag, tally = [], Counter()
    print("reports/lab5_diagnosis.json not found - run the cell above (uncomment).")


In [ ]:
# A1 tally - fill report.md from this (or from the pareto above).
if tally:
    print("YOUR TALLY (copy into report.md A1):")
    for mode in range(1, 8):
        print(f"  mode {mode} {MODES[mode]:<20} n = {tally.get(mode, 0)}")
    top2 = [MODES[m] for m, _ in tally.most_common(2)]
    print(f"\nTop two modes: {top2[0]}, {top2[1]}  (report these in A3)")
else:
    print("no tally yet - run A1")


In [ ]:
# A2 - needs_human_check: mode 2. Open the chunks and LOOK.
# For each such case: judge whether the gold answer straddles a chunk boundary.
human_cases = [d for d in diag if "needs_human_check" in d.get("evidence", "")]
print(f"{len(human_cases)} cases need your eyes (mode 2 candidates)\n")

# YOUR TURN - fill one row per case, copy into report.md A2:
A2_ROWS = """
# | qid | boundary? Y/N | evidence (what you saw in the chunks) | my mode |
# |-----|---------------|--------------------------------------|---------|
""".strip()
print(A2_ROWS)

for d in human_cases[:10]:
    q = questions.get(d["id"], {})
    print(f"\n{d['id']}: {d['question'][:100]}")
    print("  gold answer:", (q.get("gold_answer") or "")[:160])
    print("  relevant:", q.get("relevant_docs"))
    print("  -> open those docs; did the answer fall across two chunks?")


In [ ]:
# YOUR TURN - A2 answers. Fill and copy into report.md.
# For each needs_human_check id you actually opened:
A2_ANSWER = """
Q__: boundary? ___  because: ...
Q__: boundary? ___  because: ...
(repeat for every needs_human_check case you reviewed)

Final mode for each after human review:
  Q__ -> mode ___   Q__ -> mode ___
""".strip()
print(A2_ANSWER)

# If you reassigned modes after looking, update the tally here:
# tally_final = Counter({1: n1, 2: n2, ...})
# print(pareto(tally_final))


### A3 - Pareto

Name the **two modes** that account for most failures. That pair is your
candidate focus for Parts B-C.


In [ ]:
# A3 - pareto (already printed above if diag loaded). YOUR TURN answer:
A3_ANSWER = """
Two modes accounting for most failures: mode ___ (___) and mode ___ (___)
Together they are ___ of ___ failures (___%).

Why this concentration makes sense for MY pipeline (one sentence): ...
""".strip()
print(A3_ANSWER)


## Part B - Rank by expected value (20 min)

**Why.** The largest cluster is not always the right target. A cluster of 9
whose fix costs 3x per query may lose to a cluster of 6 with a free fix.

**Write the prediction down, now, before you implement.** Being wrong is
informative and is not penalised. **Not predicting is.**


In [ ]:
# YOUR TURN - B1: ranking table. One row per non-trivial cluster.
# Est. recovery = how many of that cluster's failures you expect the fix to fix.
B1_TABLE = """
# | Cluster (mode) | n | Fix | Est. recovery | Cost delta | Latency delta | Effort |
# |----------------|---|-----|---------------|------------|---------------|--------|
# | 6 generation   |   |     |               |            |               |        |
# | 3 embedding    |   |     |               |            |               |        |
# | ...            |   |     |               |            |               |        |
""".strip()
print(B1_TABLE)


In [ ]:
# YOUR TURN - B2: pick one cluster + B3: the prediction (BEFORE you code).
B2_B3_ANSWER = """
Cluster I pick: mode ___ (___)
Fix I will implement: ...
One-sentence justification (why this cluster, not a bigger one): ...

PREDICTION (write this now, before Part C):
  "I expect this to recover N = ___ of the M = ___ failures in this cluster."
""".strip()
print(B2_B3_ANSWER)
assert "N = ___" not in B2_B3_ANSWER or True  # you must fill the N/M above
print("Fill N and M above BEFORE running Part C. Not predicting costs marks.")


## Part C - Implement the fix (70 min)

**Three rules:**

1. **One fix at a time, measured each time.**
2. **Record the failures too.** A fix that did not work, with the number, is a
   result and scores marks.
3. **Fix what the diagnosis says**, not what you reach for by reflex.

**Cost discipline:** your fix must cost **<= 2x** the Lab 4 baseline per query.

The catalogue is in RUNSHEET.md step 3. Special mention: the
**archived-document trap** (Q29-Q31) - if those still fail, the fix is metadata
filtering, free, not retrieval tuning.


In [ ]:
# YOUR TURN - C1: describe the fix you are about to implement, then do it
# in rag.py / evaluate.py / diagnose.py as appropriate (ONE change).
C1_FIX = """
Mode targeted: ___
Exact change (file + what): ...
Where the change lives (function/line): ...
Cost delta expected: <= 2x baseline? Y/N because: ...
Archived-doc trap (Q29-Q31) relevant? Y/N - if yes, that is metadata filtering.
""".strip()
print(C1_FIX)

# When the change is in, re-run ONE evaluation slice (not the full 45 yet):
#   from labs.lab4.evaluate import run_full
#   run_full(save="reports/lab5_v2_partial.json")
# Or re-run the full set at the end of Part D.


In [ ]:
# C2 - optional live check: answer a few cluster questions after the fix.
# Uncomment after your fix is in and modules are reloaded.
# import importlib
# import labs.lab4.rag, labs.lab4.evaluate
# importlib.reload(labs.lab4.rag)
# importlib.reload(labs.lab4.evaluate)
# from labs.lab4.evaluate import build_retriever
# r2 = build_retriever()
# cluster_ids = [d["id"] for d in diag if d["mode"] == <your mode>][:5]
# for qid in cluster_ids:
#     q = questions[qid]
#     a = labs.lab4.rag.answer_question(q["question"], r2)
#     print(qid, a.text[:160])
print("uncomment the block above after your Part C fix is implemented")


## Part D - Prove it (25 min)

**D1.** Before/after on the **same** questions - **every** Lab 4 metric, not
just the one you targeted.

**D2. The regression check.** Report anything that got worse, **prominently**:
refusal precision, cost, p95, kinds that were already passing.

**D3.** Re-classify remaining failures. Did the distribution shift as you
predicted? Failures moving mode 3 -> mode 6 is progress (masked problem
revealed), not a new bug.

**D4.** Next fix you would make, and what you expect it to be worth.


In [ ]:
# D1 - before/after metrics. v1 = reports/lab4.json. v2 = your re-run.
# After re-running evaluate.py --full --save reports/lab5_v2.json (or similar),
# fill the table. Falls back to placeholders if v2 not saved yet.
v1_path = ROOT / "reports/lab4.json"
v2_path = ROOT / "reports/lab5_v2.json"  # change if you saved elsewhere

def summarize(path):
    if not path.exists():
        return None
    rows = json.loads(path.read_text(encoding="utf-8"))
    ans = [r for r in rows if not r.get("unanswerable")]
    una = [r for r in rows if r.get("unanswerable")]
    refusals = [r for r in rows if r.get("refused")]
    def mean(xs):
        xs = [x for x in xs if x is not None]
        return statistics.fmean(xs) if xs else None
    import statistics
    faith = mean([r.get("faithfulness") for r in rows])
    corr = mean([r.get("correctness") for r in ans])
    cit = statistics.fmean(r.get("citations_valid", False) for r in rows)
    rec = sum(1 for r in una if r.get("refused")) / len(una) if una else 0.0
    prec = sum(1 for r in refusals if r.get("unanswerable")) / len(refusals) if refusals else 1.0
    return {"faithfulness": faith, "correctness_norm": (corr / 2 if corr is not None else None),
            "citation_validity": cit, "refusal_recall": rec, "refusal_precision": prec,
            "n": len(rows), "n_fail": sum(1 for r in rows
                if (r.get("correctness") is not None and r["correctness"] < 2)
                or not r.get("citations_valid", True))}

# fix import order
import statistics
s1 = summarize(v1_path)
s2 = summarize(v2_path)

D1_TABLE = """
# | Metric | v1 | v2 | delta |
# |--------|----|----|-------|
""".strip()
print(D1_TABLE)
if s1:
    print("v1:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in s1.items()})
else:
    print("v1 missing - run Lab 4 E1")
if s2:
    print("v2:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in s2.items()})
    for k in s1:
        if isinstance(s1[k], float) and isinstance(s2.get(k), float):
            print(f"  delta {k}: {s2[k] - s1[k]:+.4f}")
else:
    print("v2 not found at", v2_path, "- re-run full eval after your fix, save there")


In [ ]:
# YOUR TURN - D2 regression check + D3 re-class + D4 next fix.
# Be honest. Anything that got worse goes in the report prominently.
D2_D4_ANSWER = """
D2 - what got worse (or "nothing, and here is how I checked"):
  refusal precision: v1=___ v2=___  (better retrieval often -> fewer refusals
    -> can answer unanswerable confidently - watch this)
  cost / p95: v1=___ v2=___
  kinds that were already passing: ...
  OTHER regressions: ...

D3 - re-classified remaining failures (run diagnose again on v2):
  before: top modes were ___ / ___
  after:  top modes were ___ / ___
  shift matches prediction? Y/N because: ...
  (mode 3 -> mode 6 movement = masked problem revealed = progress)

D4 - next fix I would make, and what I expect it to be worth:
  ...
""".strip()
print(D2_D4_ANSWER)


In [ ]:
# YOUR TURN - D1/D2 honest summary line for the top of report.md.
SUMMARY = """
Prediction was: recover N=___ of M=___.
What actually happened: recovered ___ / M  (delta correctness = ___).
Verdict: better / worse / mixed - and the one sentence why: ...
""".strip()
print(SUMMARY)
print("A negative delta with a clear account scores full marks. Say it plainly.")


## Submit

1. If you changed `diagnose.py` (improved `answer_in_corpus`, reclassified
   modes), those changes are the graded file.
2. `reports/lab5_before_after.json` (or your v1/v2 paths documented).
3. Fill `report.md` **by hand** - never regenerate it:
   - failure tally + Pareto chart
   - Part B ranking table + one-sentence justification
   - **your prediction, and what actually happened**
   - D1 before/after table (every metric)
   - D2 regression check, **including anything that got worse**
   - D3 re-classification
   - **the fix that did not work**, with its number
4. Cost of your fix vs 2x Lab 4 baseline.

Run the check cell below.

**Also keep open:** `RUNSHEET.md`.


In [ ]:
# Submit check.
diag_path = ROOT / "reports/lab5_diagnosis.json"
lab4_path = ROOT / "reports/lab4.json"
print("lab4.json:      ", "OK" if lab4_path.exists() else "MISSING (Lab 4 prerequisite)")
print("lab5_diagnosis: ", "OK" if diag_path.exists() else "MISSING (run Part A1)")

diag_src = (ROOT / "labs/lab5/diagnose.py").read_text(encoding="utf-8")
nie = diag_src.count("raise NotImplementedError")
print(f"diagnose.py: {nie} NotImplementedError stub(s)")

# report.md should exist under labs/lab5/ or repo root - check both
for p in [ROOT / "labs/lab5/report.md", ROOT / "report.md"]:
    if p.exists():
        print(f"report: {p.relative_to(ROOT)} ({p.stat().st_size} bytes)")
        break
else:
    print("report.md not found - fill labs/lab5/report.md by hand")

print("\nDone when: diagnosis exists, prediction + actual in report.md,")
print("before/after table complete, regressions listed, fix that failed included.")
